# Alphabet Recognition Model Retraining
This notebook retrains your model with the current TensorFlow 2.21.0 to fix the quantization_config error

## Step 1: Install dependencies

In [ ]:
%pip install bidict --quiet
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from bidict import bidict
from sklearn.utils import shuffle
import os
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

## Step 2: Upload your data files
Upload `images.npy` and `labels.npy` from your `data/` folder

In [ ]:
from google.colab import files

# Create data directory
os.makedirs('data', exist_ok=True)

print("Upload your data files:")
print("1. images.npy")
print("2. labels.npy")
print()

uploaded = files.upload()

# Move files to data directory
for filename in uploaded.keys():
    if filename.endswith('.npy'):
        os.rename(filename, f'data/{filename}')
        print(f"✓ Moved {filename} to data/")

## Step 3: Configure the model

In [ ]:
# Configuration
BATCH_SIZE = 16
EPOCHS = 20
DATA_DIR = 'data'

# Encoder - must match your app
ENCODER = bidict({
    'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6,
    'G': 7, 'H': 8, 'I': 9, 'J': 10, 'K': 11, 'L': 12,
    'M': 13, 'N': 14, 'O': 15, 'P': 16, 'Q': 17, 'R': 18,
    'S': 19, 'T': 20, 'U': 21, 'V': 22, 'W': 23, 'X': 24,
    'Y': 25, 'Z': 26
})

print(f"Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Number of classes: {len(ENCODER) + 1} (26 letters + 1)")

## Step 4: Load and explore data

In [ ]:
print("Loading data...")

# Load labels and convert to numeric format
labels = np.load(os.path.join(DATA_DIR, 'labels.npy'), allow_pickle=True)
labels = np.array([ENCODER[x] for x in labels])

# Load and preprocess images
imgs = np.load(os.path.join(DATA_DIR, 'images.npy'))
imgs = imgs.astype('float32') / 255.0

# Add channel dimension if not present
if len(imgs.shape) == 3:  # (N, 50, 50) -> (N, 50, 50, 1)
    imgs = np.expand_dims(imgs, -1)

print(f"✓ Images shape: {imgs.shape}")
print(f"✓ Labels shape: {labels.shape}")
print(f"✓ Min pixel value: {imgs.min()}, Max pixel value: {imgs.max()}")

## Step 5: Visualize sample images

In [ ]:
# Visualize a few random samples
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
fig.suptitle('Sample Images from Dataset', fontsize=16)

for i, ax in enumerate(axes.flat):
    idx = np.random.randint(0, len(imgs))
    letter = ENCODER.inverse[labels[idx]]
    ax.imshow(imgs[idx, :, :, 0], cmap='gray')
    ax.set_title(f'Letter: {letter}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## Step 6: Split data into train/test

In [ ]:
# Shuffle data
labels, imgs = shuffle(labels, imgs, random_state=42)

# Split into train/test (75/25)
split = int(len(labels) * 0.75)
labels_train, labels_test = labels[:split], labels[split:]
imgs_train, imgs_test = imgs[:split], imgs[split:]

print(f"Training set: {imgs_train.shape[0]} samples")
print(f"Test set: {imgs_test.shape[0]} samples")
print(f"Input shape: {imgs_train.shape[1:]}")

## Step 7: Build the model (exact same architecture)

In [ ]:
print("Building model...")

model = keras.Sequential([
    keras.Input(shape=(50, 50, 1)),
    layers.Conv2D(256, kernel_size=5, activation='relu'),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.3),
    layers.Conv2D(512, kernel_size=5, activation='relu'),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.3),
    layers.Conv2D(1024, kernel_size=5, activation='relu'),
    layers.MaxPooling2D(pool_size=2),
    layers.Dropout(0.3),
    layers.Flatten(),
    layers.Dense(len(ENCODER) + 1, activation='softmax')
])

print(f"✓ Model created with {model.count_params():,} parameters")
model.summary()

## Step 8: Compile the model

In [ ]:
print("Compiling model...")

optimizer = keras.optimizers.Adam()
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)

print("✓ Model compiled")

## Step 9: Train the model

In [ ]:
# Early stopping callback
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=2,
    restore_best_weights=True
)

print(f"Training for up to {EPOCHS} epochs...\n")

history = model.fit(
    imgs_train, labels_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(imgs_test, labels_test),
    callbacks=[early_stopping],
    verbose=1
)

print("\n✓ Training complete!")

## Step 10: Evaluate the model

In [ ]:
print("Evaluating model on test set...\n")

loss, accuracy = model.evaluate(imgs_test, labels_test, verbose=0)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

## Step 11: Plot training history

In [ ]:
# Plot accuracy
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
ax1.plot(history.history['accuracy'], label='Train Accuracy')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_title('Model Accuracy')
ax1.legend()
ax1.grid(True)

# Loss
ax2.plot(history.history['loss'], label='Train Loss')
ax2.plot(history.history['val_loss'], label='Val Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.set_title('Model Loss')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Step 12: Save the model

In [ ]:
# Create models directory
os.makedirs('models', exist_ok=True)

# Save in H5 format (compatible with your Flask app)
h5_path = 'models/letter.h5'
model.save(h5_path)
print(f"✓ Model saved to {h5_path}")

# Also save in native Keras format as backup
keras_path = 'models/letter.keras'
model.save(keras_path)
print(f"✓ Model also saved to {keras_path}")

## Step 13: Download your trained model

In [ ]:
# Download the models
files.download('models/letter.h5')
print("✓ Downloaded letter.h5")

files.download('models/letter.keras')
print("✓ Downloaded letter.keras")

print("\n" + "="*50)
print("Training Complete!")
print("="*50)
print(f"\nYour trained model is ready to use!")
print(f"Replace your old models/letter.h5 with the downloaded version.")
print(f"Your Flask app will now work without the quantization_config error!")